In [117]:
!pip install -q accelerate==0.21.0 peft==0.4.0 bitsandbytes==0.40.2 transformers==4.31.0 trl==0.4.7

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  error: subprocess-exited-with-error
  
  × Building wheel for tokenizers (pyproject.toml) did not run successfully.
  │ exit code: 1
  ╰─> See above for output.
  
  note: This error originates from a subprocess, and is likely not a problem with pip.
  ERROR: Failed building wheel for tokenizers
ERROR: ERROR: Failed to build installable wheels for some pyproject.toml based projects (tokenizers)


In [118]:
!pip install -q transformers datasets peft accelerate evaluate scikit-learn

In [119]:
import torch
import numpy as np
import pandas as pd

from datasets import load_dataset

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
)

print("PyTorch Version:", torch.__version__)
print("GPU Available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

PyTorch Version: 2.11.0+cu128
GPU Available: True
GPU: Tesla T4


In [120]:
from datasets import load_dataset

In [121]:
dataset = load_dataset("cardiffnlp/tweet_eval", "sentiment")



In [122]:
dataset

DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 45615
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 12284
    })
    validation: Dataset({
        features: ['text', 'label'],
        num_rows: 2000
    })
})

In [123]:
print(dataset['train'][0])

{'text': '"QT @user In the original draft of the 7th book, Remus Lupin survived the Battle of Hogwarts. #HappyBirthdayRemusLupin"', 'label': 2}


In [124]:
from transformers import AutoTokenizer

model_name = "distilbert-base-uncased"

tokenizer = AutoTokenizer.from_pretrained(model_name)

In [125]:
def tokenize_function(example):
    return tokenizer(
        example["text"],
        truncation=True,
        padding="max_length",
        max_length=128
    )

In [126]:
tokenized_dataset = dataset.map(
    tokenize_function,
    batched=True
)

In [127]:
tokenized_dataset

DatasetDict({
    train: Dataset({
        features: ['text', 'label', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 45615
    })
    test: Dataset({
        features: ['text', 'label', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 12284
    })
    validation: Dataset({
        features: ['text', 'label', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 2000
    })
})

In [128]:
sample = tokenized_dataset["train"][0]

print("Original Tweet:")
print(dataset["train"][0]["text"])

print("\nLabel:")
print(sample["label"])

print("\nInput IDs:")
print(sample["input_ids"])

print("\nAttention Mask:")
print(sample["attention_mask"])

Original Tweet:
"QT @user In the original draft of the 7th book, Remus Lupin survived the Battle of Hogwarts. #HappyBirthdayRemusLupin"

Label:
2

Input IDs:
[101, 1000, 1053, 2102, 1030, 5310, 1999, 1996, 2434, 4433, 1997, 1996, 5504, 2338, 1010, 2128, 7606, 11320, 8091, 5175, 1996, 2645, 1997, 27589, 18367, 2015, 1012, 1001, 3407, 17706, 2705, 10259, 28578, 2271, 7630, 8091, 1000, 102, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]

Attention Mask:
[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 

In [129]:
from transformers import AutoModelForSequenceClassification

model = AutoModelForSequenceClassification.from_pretrained(
    "distilbert-base-uncased",
    num_labels=3
)

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
classifier.weight       | MISSING    | 
pre_classifier.weight   | MISSING    | 
pre_classifier.bias     | MISSING    | 
classifier.bias         | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [130]:
print(model)

DistilBertForSequenceClassification(
  (distilbert): DistilBertModel(
    (embeddings): Embeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (transformer): Transformer(
      (layer): ModuleList(
        (0-5): 6 x TransformerBlock(
          (attention): DistilBertSelfAttention(
            (q_lin): Linear(in_features=768, out_features=768, bias=True)
            (k_lin): Linear(in_features=768, out_features=768, bias=True)
            (v_lin): Linear(in_features=768, out_features=768, bias=True)
            (out_lin): Linear(in_features=768, out_features=768, bias=True)
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (sa_layer_norm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
          (ffn): FFN(
            (dropout): Dropout(p=0.1, inplace=False)


In [131]:
for name, module in model.named_modules():
    if "attention" in name.lower():
        print(name)

distilbert.transformer.layer.0.attention
distilbert.transformer.layer.0.attention.q_lin
distilbert.transformer.layer.0.attention.k_lin
distilbert.transformer.layer.0.attention.v_lin
distilbert.transformer.layer.0.attention.out_lin
distilbert.transformer.layer.0.attention.dropout
distilbert.transformer.layer.1.attention
distilbert.transformer.layer.1.attention.q_lin
distilbert.transformer.layer.1.attention.k_lin
distilbert.transformer.layer.1.attention.v_lin
distilbert.transformer.layer.1.attention.out_lin
distilbert.transformer.layer.1.attention.dropout
distilbert.transformer.layer.2.attention
distilbert.transformer.layer.2.attention.q_lin
distilbert.transformer.layer.2.attention.k_lin
distilbert.transformer.layer.2.attention.v_lin
distilbert.transformer.layer.2.attention.out_lin
distilbert.transformer.layer.2.attention.dropout
distilbert.transformer.layer.3.attention
distilbert.transformer.layer.3.attention.q_lin
distilbert.transformer.layer.3.attention.k_lin
distilbert.transformer.la

In [132]:
for name, module in model.named_modules():
    if "q_lin" in name or "v_lin" in name:
        print(name)

distilbert.transformer.layer.0.attention.q_lin
distilbert.transformer.layer.0.attention.v_lin
distilbert.transformer.layer.1.attention.q_lin
distilbert.transformer.layer.1.attention.v_lin
distilbert.transformer.layer.2.attention.q_lin
distilbert.transformer.layer.2.attention.v_lin
distilbert.transformer.layer.3.attention.q_lin
distilbert.transformer.layer.3.attention.v_lin
distilbert.transformer.layer.4.attention.q_lin
distilbert.transformer.layer.4.attention.v_lin
distilbert.transformer.layer.5.attention.q_lin
distilbert.transformer.layer.5.attention.v_lin


In [133]:
!pip install peft

In [134]:
from peft import LoraConfig, get_peft_model, TaskType

In [135]:
lora_config = LoraConfig(
    task_type=TaskType.SEQ_CLS,
    r=4,
    lora_alpha=16,
    target_modules=["q_lin", "v_lin"],
    lora_dropout=0.1,
    bias="none"
)

In [136]:
!pip install --upgrade torchao

In [137]:
model = get_peft_model(model, lora_config)

instaed of updating all paramnetrs we just insert subset of th eparameters


In [138]:
model.print_trainable_parameters()

trainable params: 666,627 || all params: 67,622,406 || trainable%: 0.9858


In [139]:
for name, param in model.named_parameters():
    if param.requires_grad:
        print(name)

base_model.model.distilbert.transformer.layer.0.attention.q_lin.lora_A.default.weight
base_model.model.distilbert.transformer.layer.0.attention.q_lin.lora_B.default.weight
base_model.model.distilbert.transformer.layer.0.attention.v_lin.lora_A.default.weight
base_model.model.distilbert.transformer.layer.0.attention.v_lin.lora_B.default.weight
base_model.model.distilbert.transformer.layer.1.attention.q_lin.lora_A.default.weight
base_model.model.distilbert.transformer.layer.1.attention.q_lin.lora_B.default.weight
base_model.model.distilbert.transformer.layer.1.attention.v_lin.lora_A.default.weight
base_model.model.distilbert.transformer.layer.1.attention.v_lin.lora_B.default.weight
base_model.model.distilbert.transformer.layer.2.attention.q_lin.lora_A.default.weight
base_model.model.distilbert.transformer.layer.2.attention.q_lin.lora_B.default.weight
base_model.model.distilbert.transformer.layer.2.attention.v_lin.lora_A.default.weight
base_model.model.distilbert.transformer.layer.2.attent

In [140]:
for name, param in model.named_parameters():
    if "q_lin" in name:
        print(f"{name:90}  requires_grad={param.requires_grad}")

base_model.model.distilbert.transformer.layer.0.attention.q_lin.base_layer.weight           requires_grad=False
base_model.model.distilbert.transformer.layer.0.attention.q_lin.base_layer.bias             requires_grad=False
base_model.model.distilbert.transformer.layer.0.attention.q_lin.lora_A.default.weight       requires_grad=True
base_model.model.distilbert.transformer.layer.0.attention.q_lin.lora_B.default.weight       requires_grad=True
base_model.model.distilbert.transformer.layer.1.attention.q_lin.base_layer.weight           requires_grad=False
base_model.model.distilbert.transformer.layer.1.attention.q_lin.base_layer.bias             requires_grad=False
base_model.model.distilbert.transformer.layer.1.attention.q_lin.lora_A.default.weight       requires_grad=True
base_model.model.distilbert.transformer.layer.1.attention.q_lin.lora_B.default.weight       requires_grad=True
base_model.model.distilbert.transformer.layer.2.attention.q_lin.base_layer.weight           requires_grad=Fa

In [141]:
from transformers import DataCollatorWithPadding

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

In [142]:
!pip install evaluate

In [143]:
import evaluate
import numpy as np

accuracy = evaluate.load("accuracy")
precision = evaluate.load("precision")
recall = evaluate.load("recall")
f1 = evaluate.load("f1")

In [144]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred

    predictions = np.argmax(logits, axis=-1)

    acc = accuracy.compute(
        predictions=predictions,
        references=labels
    )

    prec = precision.compute(
        predictions=predictions,
        references=labels,
        average="weighted"
    )

    rec = recall.compute(
        predictions=predictions,
        references=labels,
        average="weighted"
    )

    f1_score = f1.compute(
        predictions=predictions,
        references=labels,
        average="weighted"
    )

    return {
        "accuracy": acc["accuracy"],
        "precision": prec["precision"],
        "recall": rec["recall"],
        "f1": f1_score["f1"]
    }
    predictions = np.argmax(logits, axis=-1)

In [145]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="./results",

    eval_strategy="epoch",
    save_strategy="epoch",

    learning_rate=2e-4,

    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,

    num_train_epochs=2,

    weight_decay=0.01,

    logging_steps=50,

    load_best_model_at_end=True,

    metric_for_best_model="f1",

    greater_is_better=True,

    report_to="none"
)

In [146]:
from transformers import Trainer


In [147]:

trainer = Trainer(
    model=model,
    args=training_args,

    train_dataset=tokenized_dataset["train"],
    eval_dataset=tokenized_dataset["validation"],
    processing_class=tokenizer,

    data_collator=data_collator,

    compute_metrics=compute_metrics,
)

In [148]:
trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1
1,0.686954,0.658285,0.704500,0.714953,0.704500,0.705048
2,0.633118,0.649732,0.714000,0.715946,0.714000,0.714807


TrainOutput(global_step=5702, training_loss=0.6686359434243212, metrics={'train_runtime': 762.0005, 'train_samples_per_second': 119.724, 'train_steps_per_second': 7.483, 'total_flos': 3068011055324160.0, 'train_loss': 0.6686359434243212, 'epoch': 2.0})

In [155]:
import torch

# Label mapping for tweet_eval/sentiment
label_map = {
    0: "Negative",
    1: "Neutral",
    2: "Positive"
}

def predict_sentiment(text):
    model.eval()

    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        padding=True,
        max_length=128
    )

    # Move inputs to the same device as the model
    device = model.device
    inputs = {k: v.to(device) for k, v in inputs.items()}

    with torch.no_grad():
        outputs = model(**inputs)

    logits = outputs.logits

    probabilities = torch.softmax(logits, dim=1)

    prediction = torch.argmax(probabilities, dim=1).item()

    confidence = probabilities[0][prediction].item()

    return {
        "Tweet": text,
        "Predicted Class": prediction,
        "Sentiment": label_map[prediction],
        "Confidence": confidence
    }

In [156]:
predict_sentiment("I absolutely love this phone!")

{'Tweet': 'I absolutely love this phone!',
 'Predicted Class': 2,
 'Sentiment': 'Positive',
 'Confidence': 0.9989866614341736}

In [157]:

predict_sentiment("This movie was terrible.")


{'Tweet': 'This movie was terrible.',
 'Predicted Class': 0,
 'Sentiment': 'Negative',
 'Confidence': 0.9063386917114258}

In [158]:
predict_sentiment("The weather is okay today.")

{'Tweet': 'The weather is okay today.',
 'Predicted Class': 2,
 'Sentiment': 'Positive',
 'Confidence': 0.8217127323150635}

In [159]:
test_results = trainer.evaluate(tokenized_dataset["test"])

print(test_results)

Training Loss,Validation Loss,Epoch,Accuracy,Precision,Recall,F1
0.633118,0.701404,2,0.681863,0.685497,0.681863,0.682301


{'eval_loss': 0.7014035582542419, 'eval_accuracy': 0.6818625854770433, 'eval_precision': 0.6854966226674784, 'eval_recall': 0.6818625854770433, 'eval_f1': 0.6823006811325459}


In [160]:
print("=" * 40)
print("LoRA Model Performance on Test Set")
print("=" * 40)

print(f"Test Loss      : {test_results['eval_loss']:.4f}")
print(f"Accuracy       : {test_results['eval_accuracy']:.4f}")
print(f"Precision      : {test_results['eval_precision']:.4f}")
print(f"Recall         : {test_results['eval_recall']:.4f}")
print(f"F1 Score       : {test_results['eval_f1']:.4f}")

LoRA Model Performance on Test Set
Test Loss      : 0.7014
Accuracy       : 0.6819
Precision      : 0.6855
Recall         : 0.6819
F1 Score       : 0.6823
